A. Setup + imports

In [1]:
import os
from pathlib import Path
# Set working directory to the parent of the notebooks folder (repo root)
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    repo_root = notebook_dir.parent
    os.chdir(repo_root)
    import sys
    sys.path.insert(0, str(repo_root))
else:
    print('Warning: Not in notebooks folder, adjust manually')

In [2]:
import json
from pathlib import Path
import pandas as pd
from src.extract import stub_extract
from src.validate import enforce_evidence
from src.checklist import stub_checklist
from src.tag import stub_tag
from src.aggregate import weekly_aggregate
from src.detect import detect_anomalies
from src.sitrep import stub_sitrep
from src.io import save_jsonl, save_csv

OUT = Path('data_synth')
OUT.mkdir(exist_ok=True)

B. Create tiny toy inputs (5 encounters)

In [ ]:
encounters = [
    {
        'encounter_id': 'e1',
        'week_id': 1,
        'date': '2024-01-01',
        'location_id': 'locA',
        'note_text': 'Patient reports fever and cough for 2 days.'
    },
    {
        'encounter_id': 'e2',
        'week_id': 1,
        'date': '2024-01-02',
        'location_id': 'locA',
        'note_text': 'Several episodes of watery diarrhea at home.'
    },
    {
        'encounter_id': 'e3',
        'week_id': 2,
        'date': '2024-01-08',
        'location_id': 'locA',
        'note_text': 'No fever, but mild cough.'
    },
    {
        'encounter_id': 'e4',
        'week_id': 2,
        'date': '2024-01-09',
        'location_id': 'locB',
        'note_text': 'Fever noted; patient denies cough.'
    },
    {
        'encounter_id': 'e5',
        'week_id': 3,
        'date': '2024-01-15',
        'location_id': 'locA',
        'note_text': 'Patient with watery stools and abdominal cramps; diarrhea persists.'
    },
    {
        'encounter_id': 'e6',
        'week_id': 6,
        'date': '2024-02-05',
        'location_id': 'locA',
        'note_text': 'Patient has fever.'
    },
    {
        'encounter_id': 'e7',
        'week_id': 6,
        'date': '2024-02-06',
        'location_id': 'locA',
        'note_text': 'Patient has fever.'
    },
    {
        'encounter_id': 'e8',
        'week_id': 6,
        'date': '2024-02-07',
        'location_id': 'locA',
        'note_text': 'Patient has fever.'
    },
    {
        'encounter_id': 'e9',
        'week_id': 6,
        'date': '2024-02-08',
        'location_id': 'locA',
        'note_text': 'Patient has fever.'
    },
    {
        'encounter_id': 'e10',
        'week_id': 6,
        'date': '2024-02-09',
        'location_id': 'locA',
        'note_text': 'Patient has fever.'
    },
    {
    'encounter_id': 'e11',
    'week_id': 7,
    'date': '2024-02-12',
    'location_id': 'locA',
    'note_text': 'Patient TRIGGER_BAD_EVIDENCE has cough.'
}
]
len(encounters)

10

C. Stub extractor: function stub_extract(note_text) -> encounter dict

In [4]:
# Run extractor over notes
processed = []
for e in encounters:
    extracted = stub_extract(e['note_text'])
    processed.append({
        'encounter_id': e['encounter_id'],
        'week_id': e['week_id'],
        'date': e['date'],
        'location_id': e['location_id'],
        'note_text': e['note_text'],
        'extracted': extracted,
    })
processed[0]

{'encounter_id': 'e1',
 'week_id': 1,
 'date': '2024-01-01',
 'location_id': 'locA',
 'note_text': 'Patient reports fever and cough for 2 days.',
 'extracted': {'symptoms': {'fever': {'value': 'yes',
    'evidence_quote': 'fever'},
   'cough': {'value': 'yes', 'evidence_quote': 'cough'},
   'watery_diarrhea': {'value': 'unknown', 'evidence_quote': None}},
  'onset': None,
  'severity': None}}

D. Evidence validator: enforce_evidence(encounter, note_text)

In [5]:
# Enforce evidence: downgrade any non-verified yes/no claims to 'unknown'
for p in processed:
    p['extracted'] = enforce_evidence(p['extracted'], p['note_text'])
processed[0]

{'encounter_id': 'e1',
 'week_id': 1,
 'date': '2024-01-01',
 'location_id': 'locA',
 'note_text': 'Patient reports fever and cough for 2 days.',
 'extracted': {'symptoms': {'fever': {'value': 'yes',
    'evidence_quote': 'fever'},
   'cough': {'value': 'yes', 'evidence_quote': 'cough'},
   'watery_diarrhea': {'value': 'unknown', 'evidence_quote': None}},
  'onset': None,
  'severity': None}}

E. Checklist stub: stub_checklist(encounter, note_text) -> up to 3 missing questions

In [6]:
for p in processed:
    p['checklist'] = stub_checklist(p['extracted'], p['note_text'])
processed[1]

{'encounter_id': 'e2',
 'week_id': 1,
 'date': '2024-01-02',
 'location_id': 'locA',
 'note_text': 'Several episodes of watery diarrhea at home.',
 'extracted': {'symptoms': {'fever': {'value': 'unknown',
    'evidence_quote': None},
   'cough': {'value': 'unknown', 'evidence_quote': None},
   'watery_diarrhea': {'value': 'yes', 'evidence_quote': 'watery diarrhea'}},
  'onset': None,
  'severity': None},
 'checklist': ['Is there fever?', 'Is there cough?']}

F. Syndrome tag stub: stub_tag(encounter, note_text) -> allowed tags

In [7]:
events = []
for p in processed:
    tag = stub_tag(p['extracted'], p['note_text'])
    events.append({'encounter_id': p['encounter_id'], 'week_id': p['week_id'], 'location_id': p['location_id'], 'syndrome_tag': tag})
events[:3]

[{'encounter_id': 'e1',
  'week_id': 1,
  'location_id': 'locA',
  'syndrome_tag': 'respiratory_fever'},
 {'encounter_id': 'e2',
  'week_id': 1,
  'location_id': 'locA',
  'syndrome_tag': 'acute_watery_diarrhea'},
 {'encounter_id': 'e3',
  'week_id': 2,
  'location_id': 'locA',
  'syndrome_tag': 'respiratory_fever'}]

G. Build events table: dataframe with encounter_id, week_id, location_id, syndrome_tag

In [8]:
events_df = pd.DataFrame(events)
events_df.to_csv(OUT / 'events.csv', index=False)
events_df

,encounter_id,week_id,location_id,syndrome_tag
0,e1,1,locA,respiratory_fever
1,e2,1,locA,acute_watery_diarrhea
2,e3,2,locA,respiratory_fever
3,e4,2,locB,respiratory_fever
4,e5,3,locA,acute_watery_diarrhea
5,e6,6,locA,respiratory_fever
6,e7,6,locA,respiratory_fever
7,e8,6,locA,respiratory_fever
8,e9,6,locA,respiratory_fever
9,e10,6,locA,respiratory_fever


H. Weekly aggregation: counts by (week_id, location_id, syndrome_tag)

In [9]:
weekly = weekly_aggregate(events_df)
weekly.to_csv(OUT / 'weekly_counts.csv', index=False)
weekly

,week_id,location_id,syndrome_tag,count
0,1,locA,acute_watery_diarrhea,1
1,1,locA,respiratory_fever,1
2,2,locA,respiratory_fever,1
3,2,locB,respiratory_fever,1
4,3,locA,acute_watery_diarrhea,1
5,6,locA,respiratory_fever,5


I. Deterministic anomaly detection using the frozen rule

In [10]:
anomalies = detect_anomalies(weekly)
anomalies.to_csv(OUT / 'anomalies.csv', index=False)
anomalies

,week_id,location_id,syndrome_tag,count,baseline_mean,baseline_window_size
0,6.0,locA,respiratory_fever,5.0,0.25,4


J. SITREP generator: stub_sitrep(anomalies_df)

In [11]:
sitrep = stub_sitrep(anomalies)
print(sitrep)

Weekly SITREP:

Location locA: 5 respiratory_fever cases in week 6 (baseline 0.2)

Note: these are signals only and NOT DIAGNOSIS.


K. Save outputs to data_synth/ and print examples

In [12]:
# Save encounters jsonl with extracted and checklist fields
out_records = []
for p in processed:
    out_records.append({
        'encounter_id': p['encounter_id'],
        'week_id': p['week_id'],
        'location_id': p['location_id'],
        'note_text': p['note_text'],
        'extracted': p['extracted'],
        'checklist': p.get('checklist', []),
    })
save_jsonl(OUT / 'encounters.jsonl', out_records)
save_csv(OUT / 'events.csv', events)
weekly.to_csv(OUT / 'weekly_counts.csv', index=False)
anomalies.to_csv(OUT / 'anomalies.csv', index=False)

# Print one example encounter JSON, anomalies table, SITREP
print('Example encounter JSON:')
print(json.dumps(out_records[0], indent=2, ensure_ascii=False))

print('Anomalies table:')
print(anomalies.to_string(index=False))

print('SITREP:')
print(sitrep)

Example encounter JSON:
{
  "encounter_id": "e1",
  "week_id": 1,
  "location_id": "locA",
  "note_text": "Patient reports fever and cough for 2 days.",
  "extracted": {
    "symptoms": {
      "fever": {
        "value": "yes",
        "evidence_quote": "fever"
      },
      "cough": {
        "value": "yes",
        "evidence_quote": "cough"
      },
      "watery_diarrhea": {
        "value": "unknown",
        "evidence_quote": null
      }
    },
    "onset": null,
    "severity": null
  },
  "checklist": [
    "Is there watery diarrhea?"
  ]
}
Anomalies table:
 week_id location_id      syndrome_tag  count  baseline_mean  baseline_window_size
     6.0        locA respiratory_fever    5.0           0.25                     4
SITREP:
Weekly SITREP:

Location locA: 5 respiratory_fever cases in week 6 (baseline 0.2)

Note: these are signals only and NOT DIAGNOSIS.
